## Introduction

This notebook develops user interface (UI) menus options can be used to retrieve relevant WDL tasks for use in RAG.

We need user input in order to select the right WDL tasks to build a worfkflow. We need to know what kind of input data they have (e.g. paired-end DNA FASTQs), their analysis goals (e.g. variant calling), and if they prefer any specific tools (e.g. bwa-mem, strelka). Our ChromaDB RAG database 

We want to avoid free-text input for many reasons, inclding to prevent users from generating toxic output or leaking protected data. Therefore, we will give users preset options that can be mapped to metadata keywords in our database. In a point-and-click interface these would be dropdown menus. For this MVP we will use a command-line interface with numbered options.

## 1. Imports and loading database

In [1]:
import chromadb

def get_collection(chroma_dir, collection_name="wdl_tasks"):
    client = chromadb.PersistentClient(path=chroma_dir)
    return client.get_collection(name=collection_name)

collection = get_collection('../../data/chroma/')

## 2. Decide metadata to get user input for and filter on

What metadata-related information can the user provide? We should collect that and use it to keyword-filter WDL tasks.

Example of metadata for a single WDL task (some/all can be provided as context to the LLM):

```{python}
    metadatas=[{
        "tool": "strelka",
        "task": "strelka_germline",
        "topic": ["genomics", "dna_polymorphism"],
        "species": ["eukaryote"],
        "operation": "variant_calling",
        "input_sample_data_types": ["nucleic_acid_sequence_alignment", "data_index"],
        "input_sample_format_types": ["bam", "bai"],
        "output_sample_data_types": ["sequence_variations", "data_index"]
    }]
```

**I imagine we'll ask the user:**
- "What kind of sequencing data do you have?" (e.g. DNA, Bulk RNA, etc.)
- "What format is your data in?" (e.g. BAM, FASTQ, etc.)
- "What species is your data from?" (e.g. human, non-human eukaryote, etc.)
- "What kind of processing do you want done on this data?" (e.g. QC, Alignment, etc.) - NOTE: variant calling may require alignment first etc
- "Do you have any bioinformatics tool preferences?" (e.g. fastqc, bwa)

### Metadata categories relevant to the user interface:
- `tool`: User may have bioinformatics preferences (terms can be used as-is)
- `topic`: Whether the data is DNA/RNA/protein, etc.
- `species`: What species the data is from
- `operation`: What the user wants the WDL to do
- `input_sample_format_types`: Format usually tells us enough about the data type (e.g. FASTQ, BAM)

In [2]:
# Get all the metadata sections and look at first one as an example
all_metadata = collection.get(include=['metadatas'])['metadatas']
all_metadata[0]

{'operation': ['indexing'],
 'tool': 'cnvkit',
 'input_sample_data_types': ['nucleic_acid_sequence_alignment', 'data_index'],
 'output_sample_data_types': ['none'],
 'input_sample_format_types': ['bam', 'bai'],
 'species': ['human', 'eukaryote'],
 'task': 'create_reference',
 'topic': ['genomics', 'copy_number_variation']}

## 3. Get terms to be used as-is

Terms from categories:
- `tool`
- `input_sample_format_types`

First, a function to collect unique terms:

In [3]:
def get_unique_terms(category, all_meta):
    """Get all the unique terms used for a given metadata category 
    across all_meta (which should be entire database metadata)
    """
    unique_terms = set()
    for metadata in all_meta:
        terms_list = metadata.get(category, [])
        if isinstance(terms_list, list):
            unique_terms.update(terms_list)
        elif isinstance(terms_list, str):
            unique_terms.add(terms_list)
    return sorted(unique_terms)

**`tool`**

Tools we should omit from the 'bioinformatics tool preferences' menu (no choice for some operations, or not relevant to most users):
- aws-sso
- ena
- gatk (need to be task-level specific)
- gdc
- sjl
- sra


Some 'operation' options to consider:
- Download from SRA
- Download from ENA
- Convert file formats

Some 'input data type' and 'topic' related options to consider:
- Single-cell RNA
- Bulk RNA

In [11]:
tools_list = get_unique_terms("tool", all_metadata)
tools_list

['annotsv',
 'annovar',
 'aws-sso',
 'bcftools',
 'bedparse',
 'bedtools',
 'bowtie',
 'bowtie2',
 'bwa',
 'cellranger',
 'clair3',
 'cnvkit',
 'colabfold',
 'consensus',
 'deeptools',
 'deepvariant',
 'delly',
 'deseq2',
 'diamond',
 'ena',
 'esmfold',
 'fastp',
 'fastqc',
 'gatk',
 'gdc',
 'gffread',
 'glimpse2',
 'ichorcna',
 'jcast',
 'manta',
 'megahit',
 'multiqc',
 'rmats-turbo',
 'rseqc',
 'salmon',
 'samtools',
 'shapemapper',
 'sjl',
 'smoove',
 'sourmash',
 'spades',
 'sra',
 'star',
 'starling',
 'strelka',
 'trimgalore',
 'tritonnp',
 'varscan',
 'viennarna']

In [12]:
# GATK tasks a user may want to select
gatk_list = [
    'gatk: mutect2',
    'gatk: markduplicates',
    'gatk: baserecalibrator',
    'gatk: haplotypecaller',
    'gatk: fastqtosam',
    'gatk: analyzesaturationmutagenesis'
    ]
tools_to_remove = {'gatk', 'sjl'}

updated_tool_list = [t for t in tools_list if t not in tools_to_remove] + gatk_list
updated_tool_list

['annotsv',
 'annovar',
 'aws-sso',
 'bcftools',
 'bedparse',
 'bedtools',
 'bowtie',
 'bowtie2',
 'bwa',
 'cellranger',
 'clair3',
 'cnvkit',
 'colabfold',
 'consensus',
 'deeptools',
 'deepvariant',
 'delly',
 'deseq2',
 'diamond',
 'ena',
 'esmfold',
 'fastp',
 'fastqc',
 'gdc',
 'gffread',
 'glimpse2',
 'ichorcna',
 'jcast',
 'manta',
 'megahit',
 'multiqc',
 'rmats-turbo',
 'rseqc',
 'salmon',
 'samtools',
 'shapemapper',
 'smoove',
 'sourmash',
 'spades',
 'sra',
 'star',
 'starling',
 'strelka',
 'trimgalore',
 'tritonnp',
 'varscan',
 'viennarna',
 'gatk: mutect2',
 'gatk: markduplicates',
 'gatk: baserecalibrator',
 'gatk: haplotypecaller',
 'gatk: fastqtosam',
 'gatk: analyzesaturationmutagenesis']

In [13]:
# Make it a dict (mapping terms to themselves)
tools_dict = {tool: [tool] for tool in updated_tool_list}
tools_dict

{'annotsv': ['annotsv'],
 'annovar': ['annovar'],
 'aws-sso': ['aws-sso'],
 'bcftools': ['bcftools'],
 'bedparse': ['bedparse'],
 'bedtools': ['bedtools'],
 'bowtie': ['bowtie'],
 'bowtie2': ['bowtie2'],
 'bwa': ['bwa'],
 'cellranger': ['cellranger'],
 'clair3': ['clair3'],
 'cnvkit': ['cnvkit'],
 'colabfold': ['colabfold'],
 'consensus': ['consensus'],
 'deeptools': ['deeptools'],
 'deepvariant': ['deepvariant'],
 'delly': ['delly'],
 'deseq2': ['deseq2'],
 'diamond': ['diamond'],
 'ena': ['ena'],
 'esmfold': ['esmfold'],
 'fastp': ['fastp'],
 'fastqc': ['fastqc'],
 'gdc': ['gdc'],
 'gffread': ['gffread'],
 'glimpse2': ['glimpse2'],
 'ichorcna': ['ichorcna'],
 'jcast': ['jcast'],
 'manta': ['manta'],
 'megahit': ['megahit'],
 'multiqc': ['multiqc'],
 'rmats-turbo': ['rmats-turbo'],
 'rseqc': ['rseqc'],
 'salmon': ['salmon'],
 'samtools': ['samtools'],
 'shapemapper': ['shapemapper'],
 'smoove': ['smoove'],
 'sourmash': ['sourmash'],
 'spades': ['spades'],
 'sra': ['sra'],
 'star': ['s

In [14]:
# Now update the gatk ones (tool is actually just 'gatk')
tools_dict.update({key: ['gatk'] for key in tools_dict if key.startswith('gatk: ')})
tools_dict

{'annotsv': ['annotsv'],
 'annovar': ['annovar'],
 'aws-sso': ['aws-sso'],
 'bcftools': ['bcftools'],
 'bedparse': ['bedparse'],
 'bedtools': ['bedtools'],
 'bowtie': ['bowtie'],
 'bowtie2': ['bowtie2'],
 'bwa': ['bwa'],
 'cellranger': ['cellranger'],
 'clair3': ['clair3'],
 'cnvkit': ['cnvkit'],
 'colabfold': ['colabfold'],
 'consensus': ['consensus'],
 'deeptools': ['deeptools'],
 'deepvariant': ['deepvariant'],
 'delly': ['delly'],
 'deseq2': ['deseq2'],
 'diamond': ['diamond'],
 'ena': ['ena'],
 'esmfold': ['esmfold'],
 'fastp': ['fastp'],
 'fastqc': ['fastqc'],
 'gdc': ['gdc'],
 'gffread': ['gffread'],
 'glimpse2': ['glimpse2'],
 'ichorcna': ['ichorcna'],
 'jcast': ['jcast'],
 'manta': ['manta'],
 'megahit': ['megahit'],
 'multiqc': ['multiqc'],
 'rmats-turbo': ['rmats-turbo'],
 'rseqc': ['rseqc'],
 'salmon': ['salmon'],
 'samtools': ['samtools'],
 'shapemapper': ['shapemapper'],
 'smoove': ['smoove'],
 'sourmash': ['sourmash'],
 'spades': ['spades'],
 'sra': ['sra'],
 'star': ['s

**`input_sample_format_types`**

Formats we should omit because they're too specific or not meaningful (might be intermediate input for tasks):
- any
- bai
- binary_format
- configuration_file_format
- crai
- csi
- none
- sig
- tar_format
- tbi
- textual_format
- zip_format

In [6]:
formats_list = get_unique_terms("input_sample_format_types", all_metadata)
formats_list

['any',
 'bai',
 'bam',
 'bcf',
 'bed',
 'bigwig',
 'binary_format',
 'configuration_file_format',
 'crai',
 'cram',
 'csi',
 'csv',
 'directory',
 'fasta',
 'fastq',
 'gtf',
 'matrix',
 'none',
 'npz',
 'pileup',
 'sam',
 'sig',
 'tar_format',
 'tbi',
 'textual_format',
 'tsv',
 'vcf',
 'wig',
 'zip_format']

In [7]:
formats_to_remove = {
    'any', 'bai', 'binary_format', 'configuration_file_format', 'crai', 'csi',
    'none', 'sig', 'tar_format', 'tbi', 'textual_format', 'zip_format'}

updated_formats_list = [t for t in formats_list if t not in formats_to_remove]
updated_formats_list

['bam',
 'bcf',
 'bed',
 'bigwig',
 'cram',
 'csv',
 'directory',
 'fasta',
 'fastq',
 'gtf',
 'matrix',
 'npz',
 'pileup',
 'sam',
 'tsv',
 'vcf',
 'wig']

In [19]:
# Make it a dict (mapping terms to themselves)
input_format_dict = {fmt: [fmt] for fmt in updated_formats_list}
input_format_dict

{'bam': ['bam'],
 'bcf': ['bcf'],
 'bed': ['bed'],
 'bigwig': ['bigwig'],
 'cram': ['cram'],
 'csv': ['csv'],
 'directory': ['directory'],
 'fasta': ['fasta'],
 'fastq': ['fastq'],
 'gtf': ['gtf'],
 'matrix': ['matrix'],
 'npz': ['npz'],
 'pileup': ['pileup'],
 'sam': ['sam'],
 'tsv': ['tsv'],
 'vcf': ['vcf'],
 'wig': ['wig']}

## 4. Map metadata terms to user-friendly ones where needed

Most metadata terms are from the EDAM ontology (to be consistent and descriptive). However, these are not terms researchers would use in daily life (e.g. "nucleic acid sequence alignment"). We need to come up with user-friendly terms for:

- `species` (not using EDAM)
- `topic`
- `operation`

**`species`**

In [8]:
species_list = get_unique_terms("species", all_metadata)
species_list

['eukaryote', 'human', 'prokaryote', 'virus']

In [21]:
# Let's be specific about the eukaryote in our dictionary
species_dict = {'non-human eukaryote': ['eukaryote'], 
                'human': ['human'], 
                'prokaryote': ['prokaryote'],
                'virus': ['virus']}

**`topic`**

Map topic keywords to intput data being DNA, RNA, or protein. `topic` will also be used with `operation` below to filter tasks by analysis goal.

In [10]:
# Look at what we're working with
topic_list = get_unique_terms("topic", all_metadata)
topic_list

['any',
 'copy_number_variation',
 'data_quality_management',
 'dna_mutation',
 'dna_packaging',
 'dna_polymorphism',
 'epigenomics',
 'gene_expression',
 'genomics',
 'mapping',
 'metagenomics',
 'nucleic_acid_structure_analysis',
 'protein_disordered_structure',
 'protein_expression',
 'protein_structure_analysis',
 'proteomics',
 'public_health_and_epidemiology',
 'ribosome_profiling',
 'rna_splicing',
 'sequence_assembly',
 'sequence_features',
 'sequencing',
 'structural_variation',
 'transcriptomics']

Create a dictionary of mappings.

In [ ]:
# Group topics together with the kind of input data they apply to
data_type_to_topic = {
    'dna': ['any', 'data_quality_management', 'sequencing', 'genomics', 
            'epigenomics', 'dna_packaging', 'dna_mutation', 
            'dna_polymorphism', 'metagenomics', 'copy_number_variation', 
            'nucleic_acid_structure_analysis', 'structural_variation', 
            'sequence_assembly', 'mapping', 'sequence_features'],
    'rna': ['any', 'data_quality_management', 'sequencing', 'transcriptomics', 
            'gene_expression', 'rna_splicing', 'ribosome_profiling', 'mapping',
            'sequence_features'],
    'protein': ['any', 'data_quality_management', 'sequencing', 'proteomics', 
                'protein_disordered_structure', 'protein_expression', 
                'protein_structure_analysis', 'mapping']
}

**`operation`**

User interface terms should allow user to convey their analysis goal(s). There will be overlap with topics also.

In [12]:
# Refresher on the topic terms:
topic_list

['any',
 'copy_number_variation',
 'data_quality_management',
 'dna_mutation',
 'dna_packaging',
 'dna_polymorphism',
 'epigenomics',
 'gene_expression',
 'genomics',
 'mapping',
 'metagenomics',
 'nucleic_acid_structure_analysis',
 'protein_disordered_structure',
 'protein_expression',
 'protein_structure_analysis',
 'proteomics',
 'public_health_and_epidemiology',
 'ribosome_profiling',
 'rna_splicing',
 'sequence_assembly',
 'sequence_features',
 'sequencing',
 'structural_variation',
 'transcriptomics']

In [13]:
# Look at the operation terms
operation_list = get_unique_terms("operation", all_metadata)
operation_list

['aggregation',
 'alternative_splicing_prediction',
 'annotation',
 'copy_number_variation_detection',
 'data_deposition',
 'data_filtering',
 'data_formatting',
 'data_handling',
 'data_retrieval',
 'file_handling',
 'indel_detection',
 'indexing',
 'mapping',
 'nucleic_acid_structure_analysis',
 'protein_structure_prediction',
 'quality_control',
 'quantification',
 'rna_secondary_structure_prediction',
 'rna_seq_quantification',
 'sequence_alignment',
 'sequence_assembly',
 'sequence_classification',
 'sequence_conversion',
 'sequence_trimming',
 'sequencing_quality_control',
 'splitting',
 'statistical_calculation',
 'variant_calling',
 'visualisation']

In [ ]:
# Map analysis goal phrases to operation and topic terms
# Make sure 'any' in all topic lists so we're not preemptively excluding those

analysis_goals = {
    'call variants (SNPs and indels)': {
        'operation': ['variant_calling', 'indel_detection'],
        'topic': ['any', 'dna_mutation', 'dna_polymorphism', 'genomics']
    },
    'call variants (structural)': {
        'operation': ['variant_calling'],
        'topic': ['any', 'structural_variation']
    },
    'call variants (copy number)': {
        'operation': ['copy_number_variation_detection'],
        'topic': ['any', 'copy_number_variation']
    },
    'align to reference': {
        'operation': ['sequence_alignment', 'mapping', 'indexing'],
        'topic': ['any', 'mapping', 'sequencing']
    },
    'assemble sequences': {
        'operation': ['sequence_assembly'],
        'topic': ['any', 'sequence_assembly']
    },
    'download data': {
        'operation': ['data_retrieval'],
        'topic': ['any']
    },
    'upload data': {
        'operation': ['data_deposition'],
        'topic': ['any']
    },
    'perform quality control': {
        'operation': ['quality_control', 'sequencing_quality_control', 'sequence_trimming', 'data_filtering', 'visualisation'],
        'topic': ['any', 'data_quality_management']
    },
    'annotate variants': {
        'operation': ['annotation'],
        'topic': ['any', 'genomics']
    },
    'annotate sequence features': {
        'operation': ['annotation'],
        'topic': ['any', 'sequence_features']
    },
    'convert file types': {
        'operation': ['sequence_conversion', 'data_formatting'],
        'topic': ['any']
    },
    'combine files': {
        'operation': ['aggregation'],
        'topic': ['any']
    },
    'manipulate files': {
        'operation': ['file_handling', 'data_handling', 'splitting'],
        'topic': ['any']
    },
    'measure gene expression': {
        'operation': ['rna_seq_quantification', 'quantification'],
        'topic': ['any', 'gene_expression', 'transcriptomics', 'ribosome_profiling']
    },
    'perform single-cell analysis': {
        'operation': ['rna_seq_quantification'],
        'topic': ['any', 'gene_expression', 'transcriptomics']
    },
    'predict protein structure': {
        'operation': ['protein_structure_prediction'],
        'topic': ['any', 'protein_structure_analysis', 'protein_disordered_structure']
    },
    'predict alternative splicing': {
        'operation': ['alternative_splicing_prediction'],
        'topic': ['any', 'rna_splicing']
    },
    'analyze nucleic acid structure': {
        'operation': ['nucleic_acid_structure_analysis', 'rna_secondary_structure_prediction'],
        'topic': ['any', 'nucleic_acid_structure_analysis']
    },
    'classify sequences': {
        'operation': ['sequence_classification'],
        'topic': ['any', 'metagenomics']
    },
    'analyze epigenomics': {
        'operation': ['statistical_calculation'],
        'topic': ['any', 'epigenomics', 'dna_packaging']
    },
    'analyze proteomics': {
        'operation': ['indexing', 'alignemnt'],
        'topic': ['any', 'proteomics', 'protein_expression']
    },
    'analyze pathogen genomics': {
        'operation': ['public_health_and_epidemiology'],
        'topic': ['any', 'public_health_and_epidemiology']
    },
}

## 5. Write code to present menu options to user

**I imagine we'll ask the user:**
- "What kind of sequencing data do you have?" (e.g. DNA, RNA, etc.)
- "What format is your data in?" (e.g. BAM, FASTQ, etc.)
- "What species is your data from?" (e.g. human, non-human eukaryote, etc.)
- "What kind of processing do you want done on this data?" (e.g. QC, Alignment, etc.) - NOTE: variant calling may require alignment first etc
- "Do you have any bioinformatics tool preferences?" (e.g. fastqc, bwa)

In [ ]:
# Example inputs
prompt = "What kind of sequencing data do you have? (enter numbers separated by commas"
term_dict = data_type_to_topic

# Don't run within a notebook, it will hang because waiting for user input
def get_terms_from_user(term_dict: dict[str, list[str]], prompt: str) -> set[str]:
    """Present term_dict keys as menu options, then use to select values (terms)."""

    # Number each menu option based on its index and display with the prompt
    options = list(term_dict.keys())
    print(prompt)
    for i, option in enumerate(options, 1):
        print(f"  {i}. {option.upper()}")

    # Process user input to get the indices
    raw_input = input("> ").strip()
    indices = [int(x) for x in raw_input.split(",")]

    # Use the indices/keys to grab all the terms (values) from the dictionary
    terms = []
    for idx in indices:
        terms_to_add = term_dict[options[idx]]
        terms += terms_to_add

    # Return a list of unique values
    return list(set(terms))

Can't run functions needing command-line input within this notebook. See `wdl_writer/user_interface.py`. Will get user-facing menu similar to this:

```
What kind of sequencing data do you have? (enter numbers separated by commas)
  1. DNA
  2. RNA
  3. PROTEIN
```

## 6. Decide logic that connects the inputs to each other

For example, based on format what operations do they need?

I think the answer here is to use the filtering to narrow down the tasks, and then have the LLM determine, based on retrieved task metadata, if they can be chained together at all and what additional tasks may need to be used.

This means that at the user-interface phase all we should do is use the filtering logic to retrieve task names, and if no tasks can be retrieved because of incompatible inputs report that to the user. For example, if they have protein data and want to do variant calling no tasks would show up.

**Certainly the terms should be packaged together somehow to be given to a function for keyword filtering:**

In [ ]:
# An example

topic_terms = ['any', 'data_quality_management', 'sequencing', 'genomics', 
            'epigenomics', 'dna_packaging', 'dna_mutation', 
            'dna_polymorphism', 'metagenomics', 'copy_number_variation', 
            'nucleic_acid_structure_analysis', 'structural_variation', 
            'sequence_assembly', 'mapping', 'sequence_features']
format_terms = ['fastq', 'fasta']
species_terms = ['eukaryote']
operation_terms = ['variant_calling', 'indel_detection']
tool_terms = ['any']


keyword_terms = {
    'topic': topic_terms, 
    'format': format_terms,
    'species': species_terms,
    'operation': operation_terms,
    'tool': tool_terms
    }

## 7. Connect inputs to the RAG database

Do keyword filtering to retrieve tasks for the LLM. For now just return the tasks that would be retrieved (can hammer out the retrieval details later, such as choosing between ties etc.). We should see if the LLM can make good decisions about tasks using their retrieved metadata. See the end of `01_ingestion.ipynb`.

In [ ]:
# This function goes in retrieval.py

my_input_data = ["nucleic_acid_sequence_alignment", "data_index"]
my_input_format = ["bam", "bai"]
my_output_data = ["sequence_variations"]

# Generate filters (will use 'and' to ensure task meets all minimum needs)
filter_input_data = [{"input_sample_data_types": {"$contains": i}} for i in my_input_data]
filter_input_format = [{"input_sample_format_types": {"$contains": i}} for i in my_input_format]
filter_output_data = [{"output_sample_data_types": {"$contains": i}} for i in my_output_data]

# Look at what we'll be filtering on
print(filter_input_data)
print(filter_input_format)
print(filter_output_data)

# Combine
full_filter = filter_input_data + filter_input_format + filter_output_data

retrieved = collection.get(where={"$and": full_filter})

retrieved['ids']